**Code commented and slightly modified by Giulia.**

The changes do _not_ affect the core calculations! I simply added explanations and/or moved things around **for the sake of readability**.

Added new version of fun `compute_month_seasonality()` -> `compute_month_seasonality_NEW()` to allow choice of calendar year for denominator
If good this function should be moved to "snt_utils.R" (or, if only used in this nb, keep here and bring out of function for readability)

-------------

## **Rainfall Seasonality**<br>
The pipeline employs a four-step hierarchical method to classify administrative units (`ADM2`) as **seasonal**, determine the **duration** of their rainy season, and identify the respective **onset month**. 

#### **1. Identify "start" month**
The first step evaluates every specific month in the dataset (per district and year) to determine if it marks the beginning of a concentrated rainfall period.<br>
**Logic**: For each month, the pipeline calculates a "forward-looking" proportion: the ratio of rainfall in the next {n}-month block (e.g., 3, 4, or 5 months) to the total rainfall of the following 12 months (rolling annual sum).<br>
**Threshold**: If this proportion exceeds the defined `threshold_for_seasonality` (e.g., **60%**), the month is flagged as a valid "start" month (`RAINFALL_{n}_MTH_ROW_SEASONALITY = 1`). This indicates that the majority of the annual rain falls within the block starting that month.

#### **2. Classify an `ADM2` as "Seasonal" or "Non-seasonal"**
The second step aggregates the month-level "start of the block" flags to determine if the **district (`ADM2`) _consistently_** experiences a rainy season (of a specific duration).<br>
**Logic**: The data is grouped by `ADM2_ID` and `MONTH` to calculate the **frequency** (proportion of `YEAR`s) that a specific month was flagged as a "start" in Step 1.<br>
**Consistency**: If this frequency exceeds the `proportion_seasonal_years_threshold` (e.g., **70%**), the district is officially classified as "Seasonal" for that block duration (`SEASONALITY_RAINFALL_{n}_MTH = 1`). This ensures that the seasonality is a recurring climatic feature rather than a one-off event.

#### **3. Determine season _duration_**
The third step **resolves cases where a district qualifies as seasonal for multiple block durations** (e.g., it meets the criteria for both 4-month and 5-month blocks).<br>
**Logic**: The pipeline compares all valid block sizes for the district and selects the minimum duration (SEASONAL_BLOCK_DURATION_RAINFALL).<br>
**Outcome**: This defines the **shortest window** in which the required proportion of annual rainfall is concentrated.

#### **4. Determine season _onset_**
The final step identifies the **_single_ official start month** for the defined season.<br>
**Logic**: Using the valid seasonal years identified in the previous steps, the pipeline calculates the **Mode** (most frequent value) of the start months.<br>
**Outcome**: The specific month that appears most frequently as the "start" across the historical data is assigned as the `SEASONAL_BLOCK_START_MONTH`.

--------------

## Parameters

(GP) ⭐ NEW PARAMETER for Step **1. Identify "start" month**: define method to calculate **denominator**

Choices:
* `FALSE` uses the original approach coded by **Iulia** (follows **WHO** recommendations): rolling 12-month forward window 
* `TRUE` uses **Bea**'s (AHADI) approach: "calendar" year (January to December)

In [ ]:
# MOVED to "new_parameter.r" to be used across seasonality notebooks

# USE_CALENDAR_YEAR_DENOMINATOR <- TRUE

# # Create suffix to be added to all exported objects (so we can compare outputs)
# if (USE_CALENDAR_YEAR_DENOMINATOR) {
#   SUFFIX <- "_calendar"
# } else {
#   SUFFIX <- "_frollsum"
# }

In [ ]:
source("~/workspace/pipelines/snt_seasonality_rainfall/code/new_parameter.r")

Original parameters

In [ ]:
# Fallback values - for local Development only!

if (!exists("minimum_month_block_size")) {
  minimum_month_block_size <- as.integer(3)
}

if (!exists("maximum_month_block_size")) {
  maximum_month_block_size <- as.integer(5)
}

if (!exists("threshold_for_seasonality")) {
  threshold_for_seasonality <- 0.6
}

if (!exists("threshold_proportion_seasonal_years")) {
  threshold_proportion_seasonal_years <- 0.7
}

In [ ]:
minimum_month_block_size <- as.integer(minimum_month_block_size)
maximum_month_block_size <- as.integer(maximum_month_block_size)

# New by GP
print(paste("Minimum month block size: ", minimum_month_block_size))
print(paste("Maximum month block size: ", maximum_month_block_size))  

In [ ]:
# GP: moved this up

possible_month_block_sizes <- as.integer(minimum_month_block_size:maximum_month_block_size)
formatted_threshold_for_seasonality <- sprintf("%d%%", round(threshold_for_seasonality * 100))

print(paste("Possible month block sizes:", paste(possible_month_block_sizes, collapse = ", ")))
print(paste("Seasonality threshold (formatted):",formatted_threshold_for_seasonality))

These 👇 are used in "**Quality check on data completeness**"

In [ ]:
minimum_periods <- as.integer(48)
maximum_proportion_missings_overall <- 0.1
maximum_proportion_missings_per_district <- 0.2

#### **Fixed routine formatting columns**

In [ ]:
# Global variables
type_of_seasonality <- "rainfall"
data_source <- 'ERA5'
original_values_col <- 'MEAN'

# space and time columns
admin_level <- 'ADM2'
admin_id_col <- paste(admin_level, toupper('id'), sep = '_')
admin_name_col <- paste(admin_level, toupper('name'), sep = '_')
year_col <- 'YEAR' # GP: this can be skipped (make more explicit) as we always use same col names across SNT toolbox
month_col <- 'MONTH' # GP: this can be skipped (make more explicit) as we always use same col names across SNT toolbox
period_cols <- c(year_col, month_col)

## Preliminaries (Set up)

#### Paths

In [ ]:
# Paths
ROOT_PATH <- '~/workspace'
CONFIG_PATH <- file.path(ROOT_PATH, 'configuration')
CODE_PATH <- file.path(ROOT_PATH, 'code')
DATA_PATH <- file.path(ROOT_PATH, 'data')
OUTPUT_DATA_PATH <- file.path(DATA_PATH, 'seasonality_rainfall')

#### Env and Packages

In [ ]:
# Global settings
options(scipen=999)

Sys.setenv(PROJ_LIB = "/opt/conda/share/proj")
Sys.setenv(GDAL_DATA = "/opt/conda/share/gdal")

In [ ]:
# Load utils
source(file.path(CODE_PATH, "snt_utils.r"))

In [ ]:
# List required pcks
required_packages <- c(
  "jsonlite",
  "data.table",
  "ggplot2",
  "fpp3",
  "arrow",
  "glue",
  "sf",
  "RColorBrewer",
  "httr",
  "reticulate"
)

# Execute function
install_and_load(required_packages)

In [ ]:
Sys.setenv(RETICULATE_PYTHON = "/opt/conda/bin/python")
reticulate::py_config()$python
openhexa <- import("openhexa.sdk")

#### Load SNT_config.json

In [ ]:
# Load SNT config
CONFIG_FILE_NAME <- "SNT_config.json"
config_json <- tryCatch({ fromJSON(file.path(CONFIG_PATH, CONFIG_FILE_NAME)) },
    error = function(e) {
        msg <- paste0("Error while loading configuration", conditionMessage(e))  
        cat(msg)   
        stop(msg) 
    })

msg <- paste0("SNT configuration loaded from  : ", file.path(CONFIG_PATH, CONFIG_FILE_NAME)) 
log_msg(msg)

# Set config variables
COUNTRY_CODE <- config_json$SNT_CONFIG$COUNTRY_CODE
era5_dataset <- config_json$SNT_DATASET_IDENTIFIERS$ERA5_DATASET_CLIMATE
dhis2_dataset <- config_json$SNT_DATASET_IDENTIFIERS$DHIS2_DATASET_FORMATTED

print(paste("Country code: ", COUNTRY_CODE))

## Load data

#### Shapes (as `spatial_data`)

In [ ]:
# Load spatial file from dataset
spatial_data_filename <- paste(COUNTRY_CODE, "shapes.geojson", sep = "_")
spatial_data <- get_latest_dataset_file_in_memory(dhis2_dataset, spatial_data_filename)
log_msg(glue("File {spatial_data_filename} successfully loaded from dataset version: {dhis2_dataset}"))

#### Rainfall (as `original_dt`)
Data produced by pipeline [D.2 ERA5 Aggregate](https://app.openhexa.org/workspaces/ner-snt-process/pipelines/d-2-era5-aggregate-eee33c/) 

Resolution:
* Spatial: `ADM2_ID`
* Temporal: `MONTH`

Summarization of values of **mm of rainfall**: 
* `MEAN`
* `MIN` & `MAX`


In [ ]:
# Load rainfall data from dataset
rainfall_data_filename <- paste(COUNTRY_CODE, "total_precipitation_monthly.parquet", sep = "_")
original_dt <- get_latest_dataset_file_in_memory(era5_dataset, rainfall_data_filename)
log_msg(glue("File {rainfall_data_filename} successfully loaded from dataset version: {era5_dataset}"))

## Create scaffold for output table

In [ ]:
# Columns formatting
admin_data <- sf::st_drop_geometry(spatial_data) # same as `select(starts_with("ADM"))`
setDT(admin_data) # convert to data.table for easier manipulation
common_cols <- names(admin_data) 

seasonality_col <- glue('SEASONALITY', toupper(type_of_seasonality), .sep = "_") 
season_duration_col <- glue('SEASONAL_BLOCK_DURATION', toupper(type_of_seasonality), .sep = "_") # input of `compute_min_seasonality_block()`
season_start_month_col <- glue('SEASONAL_BLOCK_START_MONTH', toupper(type_of_seasonality), .sep = "_")
rain_proportion_col <- 'RAIN_PROPORTION'
final_table_cols <- c(names(admin_data), seasonality_col, season_duration_col, season_start_month_col, rain_proportion_col)
print(final_table_cols)

**Create the containers for the data**<br>
Create an empty table if the analysis is stopped for lack of enough data

In [ ]:
# Create an empty table if the analysis is stopped for lack of enough data
seasonality_cols <- c(seasonality_col, season_duration_col, season_start_month_col, rain_proportion_col)
empty_dt <- copy(admin_data)[, (seasonality_cols) := NA]

## Pre-process Rainfall data 
(`original_dt` = ERA5 Rainfall data)

In [ ]:
# format table to define col types
setDT(original_dt) # ERA5 Rainfall data
integer_cols <- c(year_col, month_col)
numeric_cols <- c(original_values_col)
original_dt[, (integer_cols) := lapply(.SD, as.integer), .SDcols = integer_cols]

head(original_dt)

### Quality check on data completeness
(🤔 GP: probably legacy of working with cases, not rainfall, as ERA5 rainfall data should always be complete ... ?)

In [ ]:
# keep only the useful columns and aggregate the data on them
original_dt <- original_dt[,
                           setNames(list(sum(get(original_values_col), na.rm = TRUE)), original_values_col), 
                           by = c(admin_id_col, period_cols)
                           ]

num_periods <- make_cartesian_admin_period(original_dt, admin_id_col, year_col, month_col)[[1]]
all_rows <- make_cartesian_admin_period(original_dt, admin_id_col, year_col, month_col)[[2]]

if (num_periods < minimum_periods){    
    log_msg(glue("Data is not reliable: 
                    at least {minimum_periods} year-month periods of data are required for the case analyais; 
                    the data only contains {num_periods} periods. Abandoning analysis.")
           , level="error")
    stop("ERROR 1")
} else {
    log_msg(glue("The data contains {num_periods} unique month-year periods for each district, which is sufficient to proceed with the analysis."))
}

# inject the (possibly missing) rows into the data
original_dt <- make_full_time_space_data(
  input_dt=original_dt,
  full_rows_dt=all_rows,
  target_colname=original_values_col,
  admin_colname=admin_id_col,
  year_colname=year_col,
  month_colname=month_col)

# GP: made logic more explicit and added log messages
nr_missing_values <- nrow(original_dt[is.na(get(original_values_col)),])
threshold_nr_missing_values <- maximum_proportion_missings_overall * nrow(original_dt)

# if(nrow(original_dt[is.na(get(original_values_col)),]) > (maximum_proportion_missings_overall * nrow(original_dt))){ # GP
if(nr_missing_values > threshold_nr_missing_values){    # GP
    log_msg("There are too many missing values in the data overall. Abandoning analysis.", level="error")
    stop("ERROR 2")   
} else { # GP
    log_msg(glue("There are {nr_missing_values} missing values in the data, which is within the acceptable threshold of {threshold_nr_missing_values}. Proceeding with the analysis."))
}

In [ ]:
# Add log mesage to specify start and end of available data 
yyyymm_min <- original_dt[, min(get(year_col)*100 + get(month_col), na.rm = TRUE)]
yyyymm_max <- original_dt[, max(get(year_col)*100 + get(month_col), na.rm = TRUE)]
period_range <- c(yyyymm_min, yyyymm_max)
log_msg(glue("Rainfall data (ERA5) available for the period range {period_range[1]} to {period_range[2]}."))

### Imputation of missings

**Remove impute files (if any)**

### 🤌🏼: where are "impute files" from? Dedicated pipeline I assume ... Is this still relevant (or legacy code)?

In [ ]:
# Remove existing imputation files
filename_imputed_dt <- paste(COUNTRY_CODE, type_of_seasonality, 'imputed.csv', sep = '_')
files_in_folder <- list.files(OUTPUT_DATA_PATH, full.names = TRUE)
files_to_remove <- files_in_folder[grepl(filename_imputed_dt, basename(files_in_folder), ignore.case = TRUE)]
file.remove(files_to_remove)
print(glue("Deleted files: {str(files_to_remove)}"))

In [ ]:
# create the name of the column which will store the imputed/estimated values
imputed_col = paste(original_values_col, 'EST', sep = '_')

# if there are rows of missing data for cases, impute them (SARIMA)
if(nrow(original_dt[!is.na(get(original_values_col)),]) != nrow(original_dt)) {
    log_msg("There is missing data. Proceeding to impute them.", level="warning")
    
    # extract data on only the administrative units which have missing values for original_values_col
    missing_dt <- extract_dt_with_missings(original_dt, target_colname = original_values_col, id_colname = admin_id_col)
    missing_dt <- missing_dt[, PERIOD := make_yearmonth(year = YEAR, month = MONTH)]
    missing_dt <- missing_dt[, .SD, .SDcols = c(admin_id_col, 'PERIOD', original_values_col)]
    
    # how many rows missing for each administrative unit? if too many, then not good idea to impute
    missings_by_admin_unit <- missing_dt[, .(missing_count = sum(is.na(get(original_values_col)))), by = admin_id_col][order(-missing_count)]
    
    # if for any given admin unit, more than a given % of data is missing, there's too much to impute (maybe should be stricter - to discuss)
    if(missings_by_admin_unit[, max(missing_count)] > maximum_proportion_missings_per_district * num_periods){
      log_msg("Some administrative units have too many missing values in the target data. Abandoning analysis.", level="error")
      stop("ERROR 3")
    }
    
    # split to list per admin_unit_id, to apply SARIMA imputation on each time series (per admin unit)
    missing_districts_list <- split(missing_dt, by = admin_id_col)
    
    # seasonal ARIMA to estimate missing cases: apply function to list of data.tables with missing rows, then create data.table from result
    filled_missings_dt <- rbindlist(
    lapply(missing_districts_list,
           fill_missing_cases_ts,
           original_values_colname=original_values_col,
           estimated_values_colname=imputed_col,
           admin_colname=admin_id_col,
           period_colname='PERIOD',
           threshold_for_missing = 0.0)
    )
    
    # add the imputed ("_EST") values to the original data
    imputed_dt <- merge.data.table(original_dt, filled_missings_dt[, .SD, .SDcols = !(original_values_col)], by = c(admin_id_col, year_col, month_col), all.x = TRUE)
    
    # copy from the districts without missings;
    # if data is large, this could be made faster by only copying from the districts which are not in the missing_dt
    imputed_dt[!is.na(get(original_values_col)), (imputed_col) := get(original_values_col)]

    # Save imputed file, only if it was computed..
    fwrite(imputed_dt, file = file.path(OUTPUT_DATA_PATH, filename_imputed_dt))
    
} else {
    imputed_dt <- copy(original_dt)
    imputed_dt[, (imputed_col) := get(original_values_col)]
}

## Evaluate Seasonality

### 1. **Identify "start" month**
> _Is this **month** a **start**?_ 

("Row-Level Seasonality")

**Logic**:
* for each `MONTH`, calculate the **proportion** of **annual rainfall** falling in the next {n}-months "**block**" 
    * With {n} defined by parameter `possible_month_block_sizes`
    * With **denominator** as the **rolling sum of the next 12 months** rainfall (moving window)
* if this proportion is > ***X%***, then that month is flagged as a "**start**" (of rain season) month
    * where: ***X%*** = the value of `threshold_for_seasonality` (e.g., 60%)

The **output** of `compute_month_seasonality()` is a dt with the following cols:
* `ADM2_ID`, `YEAR`, `MONTH`
* `RAINFALL_SUM_12_MTH_FW`: the total accumulated rainfall (mm) for the **12-month** block starting from the current month
* `RAINFALL_SUM_{n}_MTH_FW`: the total accumulated rainfall (mm) for the **{n}-month** block starting from the current month
* `RAINFALL_{n}_MTH_ROW_PROP`: the **percentage** of the **annual rainfall** that falls within that specific {n}-month block.
    * Calculated as `RAINFALL_SUM_{n}_MTH_FW / RAINFALL_SUM_12_MTH_FW`
* `RAINFALL_{n}_MTH_ROW_SEASONALITY`: a **binary flag (1 or 0)** indicating whether this specific month qualifies as the start of a rainy season "block".
    * The **Logic**: It checks if the proportion calculated above is greater than or equal to the `threshold_for_seasonality` (e.g., 60%).
    * The **Interpretation**:
        * `1` (TRUE): "If the season ({n}-months block) started in month X, it would contain at least 60% of the year's total rain. Therefore, month X is a valid candidate for the start of the season."
        * `0` (FALSE): "The {n}-months block starting in month X do not contain enough of the year's total rain."

In [ ]:
# The seasonality per row (period-admin unit) -----------------------------

row_seasonality_dt <- compute_month_seasonality(
  input_dt=imputed_dt, # The cleaned and valid rainfall data
  indicator=type_of_seasonality, # "RAINFALL" (legacy parameter name, we could simply spell it out now)
  values_colname=imputed_col, # "MEAN_EST" = mean rainfall (mm). "_EST" because it can be the original value or the imputed one, depending on the case
  vector_of_durations=possible_month_block_sizes, # pipeline param (e.g. 3, 4, 5 months)
  admin_colname=admin_id_col, # ADM2_ID (also could be hard-coded as it's always gonna be at AMD2 level ... )
  year_colname=year_col, # YEAR (also could be hard-coded as it's always gonna be the same col name across SNT toolbox ...)
  month_colname=month_col, # MONTH (also could be hard-coded as it's always gonna be the same col name across SNT toolbox ...)
  proportion_threshold=threshold_for_seasonality # pipeline param (e.g. 0.6, meaning that to be considered seasonal, the n-month block should contain at least 60% of the annual rainfall)
)

head(row_seasonality_dt, 5)

In [ ]:
# GP added for data exploration

filename <- paste0('GP_row_seasonality_dt', "_frollsum", '.parquet') # This is the "original" version. Will be overwritten if same selected for `compute_month_seasonality_NEW()`, else there will be 2 files with suffixes "_frollsum" and "_calendar" respectively
write_parquet(row_seasonality_dt, file.path("~/workspace/ad_hoc_analyses/rainfall_seasonality", filename))

-------------

### ⭐ 1. (NEW) **Identify "start" month**

> _Is this **month** a **start**?_  

In `compute_month_seasonality_NEW()` can now choose alternative approach for **denominator**: "calendar" year OR "frollsum"

In [ ]:
# ⭐ GP: new version of the function using different approach for **denominator** (rest remains the same)
# ⚠️ TO MOVE to "snt_utils.r" is approved ⚠️

compute_month_seasonality_NEW <- function(input_dt, indicator, values_colname, vector_of_durations, 
                                      admin_colname = 'ADM2_ID', year_colname = 'YEAR', month_colname = 'MONTH', 
                                      proportion_threshold = 0.6, 
                                      use_calendar_year_denominator = FALSE) {
  #' create forward-looking month blocks summing values based on the WHO month-block reasoning for seasonality computation - allows for different block sizes
  #' @param input_dt an input data table (or data frame)
  #' @param indicator a string to specify the type of indicator (case/rainfall/etc. - will be added to the output variable name)
  #' @param values_colname the indicator column, on which the computations are made
  #' @param vector_of_durations the vector with the number of months in a block (3/4/5)
  #' @param admin_colname the administrative units to group 
  #' @param year_colname year grouping column
  #' @param month_colname month grouping column
  #' @param proportion_threshold the proportion of indicator which needs to occur in a block, to qualify for seasonality
  #' @param use_calendar_year_denominator Logical. If TRUE, uses the total accumulated rainfall of the current calendar year (Jan-Dec) as the denominator. If FALSE, uses the 12-month forward rolling sum.
  #' @return an output data table with the additional column

  indicator <- toupper(indicator)
  dt <- copy(as.data.table(input_dt))
   
  # ensure correct order
  dt <- dt[order(get(admin_colname), get(year_colname), get(month_colname))]
   
  # ---------------------------------------------------------
  # DENOMINATOR CALCULATION
  # ---------------------------------------------------------
  if (use_calendar_year_denominator) {
    # Alternative Approach: Total accumulated rainfall for the current calendar year (Jan-Dec)
    # Group by Admin AND Year to get the annual sum
    denominator_colname <- paste(indicator, "SUM_CALENDAR_YEAR", sep = "_")
    
    dt[, (denominator_colname) := sum(get(values_colname), na.rm = TRUE), 
       by = c(admin_colname, year_colname)]
       
  } else {
    # Original Approach: 12-month forward-looking sliding sum (left-aligned)
    denominator_colname <- paste(indicator, "SUM", 12, "MTH", "FW", sep = "_")
    
    dt[, (denominator_colname) := frollsum(get(values_colname),
                                           n = 12,
                                           align = "left",
                                           na.rm = TRUE),
       by = admin_colname]
  }
   
  # ---------------------------------------------------------
  # NUMERATOR & RATIO CALCULATIONS
  # ---------------------------------------------------------
  # numerators for each of the durations (forward-looking)
  for (n in vector_of_durations) {
    numerator_colname  <- paste(indicator, "SUM", n, "MTH", "FW", sep = "_")
    prop_name <- paste(indicator, n, "MTH", "ROW", "PROP", sep = "_")
    seasonality_colname <- paste(indicator, n, "MTH", "ROW", "SEASONALITY", sep = "_")
     
    # Calculate numerator: Rolling sum of next n months
    dt[, (numerator_colname) := frollsum(get(values_colname),
                                         n = n,
                                         align = "left",
                                         na.rm = TRUE),
       by = admin_colname]
     
    # Calculate Proportion: Numerator / Denominator (using the dynamically selected denominator)
    dt[, (prop_name) := 
          # make NA's where it would be division by zero
          fifelse(get(denominator_colname) > 0, get(numerator_colname) / get(denominator_colname), NA_real_)]
     
    # Calculate Seasonality Flag
    dt[, (seasonality_colname) := as.integer(get(denominator_colname) > 0 &
                                             get(prop_name) >= proportion_threshold)]
  }
   
  # return the data
  dt[]
}

In [ ]:
# The seasonality per row (period-admin unit) -----------------------------

row_seasonality_dt <- compute_month_seasonality_NEW(
  input_dt=imputed_dt, # The cleaned and valid rainfall data
  indicator=type_of_seasonality, # "RAINFALL" (legacy parameter name, we could simply spell it out now)
  values_colname=imputed_col, # "MEAN_EST" = mean rainfall (mm). "_EST" because it can be the original value or the imputed one, depending on the case
  vector_of_durations=possible_month_block_sizes, # pipeline param (e.g. 3, 4, 5 months)
  admin_colname=admin_id_col, # ADM2_ID (also could be hard-coded as it's always gonna be at AMD2 level ... )
  year_colname=year_col, # YEAR (also could be hard-coded as it's always gonna be the same col name across SNT toolbox ...)
  month_colname=month_col, # MONTH (also could be hard-coded as it's always gonna be the same col name across SNT toolbox ...)
  proportion_threshold=threshold_for_seasonality, # pipeline param (e.g. 0.6, meaning that to be considered seasonal, the n-month block should contain at least 60% of the annual rainfall)
  # ⭐ NEW
  use_calendar_year_denominator=USE_CALENDAR_YEAR_DENOMINATOR # Logical. If TRUE, uses the total accumulated rainfall of the current calendar year (Jan-Dec) as the denominator. If FALSE, uses the 12-month forward rolling sum.
)

head(row_seasonality_dt, 5)

In [ ]:
# GP added for data exploration

filename <- paste0('GP_row_seasonality_dt', SUFFIX, '.parquet') 
write_parquet(row_seasonality_dt, file.path("~/workspace/ad_hoc_analyses/rainfall_seasonality", filename))

--------------

### 2. **Classify an `ADM2` as "Seasonal" or "Non-seasonal"**

> _Is the **district** (ADM2) seasonal?_

("Admin-Level Seasonality")

**Logic**:
* If a specific month (e.g., January) is flagged as a "start" in > ***Y%*** of the years, the **`ADM2` is seasonal**.
    * Namely: 
        * group by `ADM2_ID` and `MONTH` then
        * count how many `RAINFALL_{n}_MTH_ROW_SEASONALITY` == 1 (sum of success). Basically: **how many times (`YEAR`s) a given `MONTH` was a "start"**
        * calculate the proportion of years and compares it to the `proportion_seasonal_years_threshold` (***Y%***, e.g., 70%)

The **output** of `process_seasonality()` is a dt with the following cols:
* `ADM2_ID`
* `PROP_SEASONAL_RAINFALL_{n}_MTH`: the **consistency** (frequency) of a specific `MONTH` being the ***start*** of the rainy season (for a given ADM2). 
    * Basically, _how often (proportion of years) was this month the start of a {n}-months seasonal block?_
* `SEASONALITY_RAINFALL_{n}_MTH`: a **binary flag (1 or 0)** indicating whether the `ADM2` is **considered seasonal** for a {n}-month block duration
    * `1` (Seasonal): **The `ADM2` has at least one month in the year that _consistently_** (e.g., in >70% of years) **marks the start of a "rain season"**. Where "rain season" is a {n}-month block containing the majority (e.g., > 60%) of the annual rainfall.
    * `0` (Not Seasonal): No month in the year meets the consistency criteria for that {n}-month block.

In [ ]:
# The seasonality per admin unit, irrespective of year ----------------------

seasonality_source_dt <- process_seasonality(
  input_dt=row_seasonality_dt, # Output of the previous step ("is this month a start?")
  indicator=type_of_seasonality, # "RAINFALL" (legacy parameter name, we could simply spell it out now)
  vector_of_durations=possible_month_block_sizes, # pipeline param (e.g. 3, 4, 5 months)
  admin_colname=admin_id_col, # ADM2_ID (also could be hard-coded as it's always gonna be at AMD2 level ... )
  year_colname=year_col, # YEAR (also could be hard-coded as it's always gonna be the same col name across SNT toolbox ...)
  month_colname=month_col, # MONTH (also could be hard-coded as it's always gonna be the same col name across SNT toolbox ...)
  proportion_seasonal_years_threshold=threshold_proportion_seasonal_years # pipeline param (e.g. 0.5, meaning that to be considered seasonal, at least 50% of the years should have the same seasonal pattern
)

head(seasonality_source_dt, 5)

In [ ]:
# GP: export the seasonality_source_dt as well for data exploration and debugging purposes
filename <- paste0('GP_seasonality_source_dt', SUFFIX, '.parquet')
write_parquet(seasonality_source_dt, file.path("~/workspace/ad_hoc_analyses/rainfall_seasonality", filename))

In [ ]:
# GP: keep only relevant cols: `ADM2_ID`, `SEASONALITY_RAINFALL_{n}_MTH`
#  (e.g., SEASONALITY_RAINFALL_3_MTH, SEASONALITY_RAINFALL_4_MTH, SEASONALITY_RAINFALL_5_MTH, ...)
check_pattern_seasonality <- paste0("^SEASONALITY_RAINFALL", "_[0-9]+_MTH$")
seasonality_source_dt <- seasonality_source_dt %>%
    select(all_of(admin_id_col), matches(check_pattern_seasonality))

head(seasonality_source_dt)

#### Result file

##### Make "long"

In [ ]:
seasonality_long_dt <- melt(
  seasonality_source_dt,
  id.vars = grep(check_pattern_seasonality, names(seasonality_source_dt), value = TRUE, invert = TRUE), # all cols which don't follow the pattern
  variable.name = 'MONTH_BLOCK_SIZE',
  value.name =seasonality_col
  )

head(seasonality_long_dt)

In [ ]:
seasonality_long_dt[, MONTH_BLOCK_SIZE := possible_month_block_sizes[match(MONTH_BLOCK_SIZE, grep(check_pattern_seasonality, names(seasonality_source_dt), value = TRUE))]]

# add remaining admin unit columns and save the final results
admin_seasonality_long_dt <- merge.data.table(admin_data, seasonality_long_dt, by = c(admin_id_col), all = TRUE)

head(admin_seasonality_long_dt)

In [ ]:
# order the columns
specific_cols <- setdiff(names(admin_seasonality_long_dt), names(admin_data)) # last columns
admin_seasonality_long_dt <- admin_seasonality_long_dt[, .SD, .SDcols = c(common_cols, specific_cols)]

head(admin_seasonality_long_dt)

In [ ]:
# GP: Export 
filename <- paste0('GP_admin_seasonality_long_dt', SUFFIX, '.parquet')
write_parquet(seasonality_source_dt, file.path("~/workspace/ad_hoc_analyses/rainfall_seasonality", filename))

### 3. **Determine season _duration_**
> _How long is the rain season?_

Idenitfy length (number of months) of the rain season.

**Logic**: If an `ADM2` qualifies for multiple block sizes (e.g., 3, 4, and 5 months), pick the ***shortest*** period (minimum).

<div class="alert alert-block alert-warning">
Idea: <br>
Here we should try to add the info of the <b>proportion of yearly rainfall</b> that falls in the {n}-month block!
</div>

The **output** of `compute_min_seasonality_block()` is a dt with the following cols:
* `ADM2_ID`
* `SEASONALITY_RAINFALL_{n}_MTH`: binary flag (1 or 0) for `ADM2` **seasonality**. Produced in previous step (2) and retained here
* `SEASONAL_BLOCK_DURATION_RAINFALL`: minimum duration of rainfall season

##### (Transform to wide format)

In [ ]:
seasonality_wide_dt <- compute_min_seasonality_block(
    input_dt=seasonality_source_dt, # Output of the previous step ("is ADM2 seasonal for a given {n}-months block?"), then keep only the relevant cols
    seasonality_column_pattern=check_pattern_seasonality, # '^SEASONALITY_RAINFALL_[0-9]+_MTH$'
    vector_of_possible_month_block_sizes=possible_month_block_sizes, # pipeline param (e.g. 3, 4, 5 months)
    seasonal_blocksize_colname=season_duration_col, # output colname: 'SEASONAL_BLOCK_DURATION_RAINFALL'
    valid_value = 1 # value indicating seasonality
)

head(seasonality_wide_dt)

-------------

### 3. (Simplified) **Determine season _duration_**

> _How long is the rain season?_

This does exactly the same as `compute_min_seasonality_block()`, but it's more accessible (readable) as it does not require importing a dedicated function from snt_utils.r

In [ ]:
head(admin_seasonality_long_dt)

In [ ]:
# GP simplified code: 
# same as `compute_min_seasonality_block() but without function call (seems unnecessarly complicated)

seasonality_wide_GP <- admin_seasonality_long_dt |>
filter(SEASONALITY_RAINFALL == 1) |>
group_by(ADM2_ID) |>
summarise(
    SEASONAL_BLOCK_DURATION_RAINFALL = min(MONTH_BLOCK_SIZE, na.rm = TRUE)
) 

head(seasonality_wide_GP)

In [ ]:
# Compare the two versions of the seasonality block duration 
# (the one computed with the function and the one computed with the simplified code) 
# to make sure they are the same

original <- seasonality_wide_dt |> select(ADM2_ID, SEASONAL_BLOCK_DURATION_RAINFALL) |> arrange(ADM2_ID) 
simplified <- seasonality_wide_GP |> select(ADM2_ID, SEASONAL_BLOCK_DURATION_RAINFALL) |> arrange(ADM2_ID)

# Check if any FALSE values in the comparison of the two dataframes
any(original$SEASONAL_BLOCK_DURATION_RAINFALL != simplified$SEASONAL_BLOCK_DURATION_RAINFALL)

--------------------

### 🤌🏼 What is this for??

In [ ]:
# Create a new, overall column 'SEASONALITY_' based on the values of columns in 'check_pattern_seasonality'
seasonality_pattern_cols <- grep(check_pattern_seasonality, names(seasonality_wide_dt), value = TRUE)
if (length(seasonality_pattern_cols) > 0L) {
  seasonality_wide_dt <- seasonality_wide_dt[, (seasonality_col) := ifelse(rowSums(.SD == 1, na.rm = TRUE) > 0, 1L, 0L), .SDcols = seasonality_pattern_cols]
  seasonality_wide_dt <- seasonality_wide_dt[, (seasonality_pattern_cols) := NULL]
} else {
  seasonality_wide_dt[, (seasonality_col) := NA_integer_]
}

In [ ]:
head(seasonality_wide_dt)

In [ ]:
# Compute RAIN_PROPORTION: proportion of rainfall in the seasonal block vs ANNUAL total
# Only for seasonal admin units (SEASONALITY_RAINFALL = 1)

# Step 1: Compute annual totals per admin-year from imputed data
annual_totals_dt <- imputed_dt[, .(ANNUAL_TOTAL = sum(get(imputed_col), na.rm = TRUE)), by = c(admin_id_col, year_col)]

head(annual_totals_dt)

In [ ]:
# Step 2: Function to compute proportion = max block sum / annual total
compute_rain_proportion <- function(admin_id, block_duration, row_data, annual_data, admin_col, year_column) {
  if (is.na(block_duration) || is.infinite(block_duration)) return(NA_real_)
  
  # Column with block sum (N-month forward-looking sum)
  sum_col <- paste('RAINFALL_SUM', block_duration, 'MTH_FW', sep = '_')
  if (!sum_col %in% names(row_data)) return(NA_real_)
  
  admin_row_data <- row_data[get(admin_col) == admin_id]
  admin_annual_data <- annual_data[get(admin_col) == admin_id]
  if (nrow(admin_row_data) == 0 || nrow(admin_annual_data) == 0) return(NA_real_)
  
  # For each year, get max block sum (only if there are non-NA values)
  yearly_max_block <- admin_row_data[
    !is.na(get(sum_col)),
    .(max_block_sum = if (.N > 0L) max(get(sum_col), na.rm = TRUE) else NA_real_),
    by = year_column
  ]
  
  # Remove rows with NA or -Inf (from max when all values were NA)
  yearly_max_block <- yearly_max_block[is.finite(max_block_sum)]
  if (nrow(yearly_max_block) == 0) return(NA_real_)
  
  # Merge with annual totals
  merged <- merge(yearly_max_block, admin_annual_data, by = year_column)
  merged <- merged[ANNUAL_TOTAL > 0]
  if (nrow(merged) == 0) return(NA_real_)
  
  # Proportion = block sum / annual total, then average across years
  merged[, prop := max_block_sum / ANNUAL_TOTAL]
  return(mean(merged$prop, na.rm = TRUE))
}

In [ ]:
seasonality_wide_dt[, (rain_proportion_col) := mapply(
  compute_rain_proportion,
  admin_id = get(admin_id_col), 
  block_duration = get(season_duration_col), # value of col `SEASONAL_BLOCK_DURATION_RAINFALL` (e.g., 3, 4, 5 (months))
  MoreArgs = list(
    row_data = row_seasonality_dt, # output of step 1 ("is this month a start of the seasonal block?")
    annual_data = annual_totals_dt, # output of cell just above this one (annual rainfall totals per admin-year)
    admin_col = admin_id_col, # 'ADM2_ID'
    year_column = year_col # 'YEAR'
    )
)]

head(seasonality_wide_dt)

In [ ]:
seasonality_wide_dt |> filter(ADM2_ID == 'AXYBpkHRL1P') 

In [ ]:
# Set RAIN_PROPORTION to NA for non-seasonal admin units
seasonality_wide_dt[get(seasonality_col) == 0 | is.na(get(seasonality_col)), (rain_proportion_col) := NA_real_]

head(seasonality_wide_dt)

In [ ]:
head(seasonality_wide_dt)

### 4. **Determine season _onset_**

> _**When** does the rain season **start**?_

**Logic**: Find the **most frequent** start `MONTH` (Mode).

In [ ]:
# # Compute SEASONAL_BLOCK_START_MONTH: first month of the seasonal block
# # Only for seasonal admin units (SEASONALITY_RAINFALL = 1)

# # Function to find the most frequent starting month for a given admin unit and block duration
# compute_start_month <- function(admin_id, block_duration, row_data, admin_col, year_column, month_column) {
#   if (is.na(block_duration) || is.infinite(block_duration)) return(NA_integer_)
  
#   # Column with row-level seasonality indicator for this block duration
#   seasonality_row_col <- paste('RAINFALL', block_duration, 'MTH_ROW_SEASONALITY', sep = '_')
#   if (!seasonality_row_col %in% names(row_data)) return(NA_integer_)
  
#   admin_row_data <- row_data[get(admin_col) == admin_id]
#   if (nrow(admin_row_data) == 0) return(NA_integer_)
  
#   # Filter rows where seasonality = 1 (this month is the start of a seasonal block)
#   seasonal_months <- admin_row_data[get(seasonality_row_col) == 1, get(month_column)]
  
#   if (length(seasonal_months) == 0) return(NA_integer_)
  
#   # Find the most frequent month (mode)
#   month_counts <- table(seasonal_months)
#   most_frequent_month <- as.integer(names(month_counts)[which.max(month_counts)])
  
#   return(most_frequent_month)
# }

# seasonality_wide_dt[, (season_start_month_col) := mapply(
#   compute_start_month,
#   admin_id = get(admin_id_col), # 'ADM2_ID'
#   block_duration = get(season_duration_col), # `SEASONAL_BLOCK_DURATION_RAINFALL` (can be hard-coded!)
#   MoreArgs = list(
#     row_data = row_seasonality_dt, # output of step 1 ("is this month a start?")
#     admin_col = admin_id_col, # 'ADM2_ID'
#     year_column = year_col,   # 'YEAR'
#     month_column = month_col  # 'MONTH'
#     )
# )]

# head(seasonality_wide_dt)

In [ ]:
# # Set SEASONAL_BLOCK_START_MONTH to NA for non-seasonal admin units
# seasonality_wide_dt[get(seasonality_col) == 0 | is.na(get(seasonality_col)), (season_start_month_col) := NA_integer_]

# head(seasonality_wide_dt)

----------------

### 4. (Commented) **Determine season _onset_**

> _**When** does the rain season **start**?_

**Logic**: Find the **most frequent** start `MONTH` (Mode).

In [ ]:
# Compute SEASONAL_BLOCK_START_MONTH: first month of the seasonal block
# Only for seasonal admin units (SEASONALITY_RAINFALL = 1)

# Function to find the most frequent starting month for a given admin unit and block duration
compute_start_month <- function(admin_id, block_duration, row_data, admin_col, year_column, month_column) {
  
  # VALIDATION: Check if the block duration is valid (not NA or Infinite). 
  # If invalid, return an integer NA immediately.
  if (is.na(block_duration) || is.infinite(block_duration)) return(NA_integer_)
  
  # DYNAMIC COLUMN GENERATION: Construct the specific column name based on the block duration.
  # Example: If block_duration is 3, this looks for 'RAINFALL_3_MTH_ROW_SEASONALITY'.
  seasonality_row_col <- paste('RAINFALL', block_duration, 'MTH_ROW_SEASONALITY', sep = '_')
  
  # SAFETY CHECK: Ensure the constructed column name actually exists in the provided dataset.
  if (!seasonality_row_col %in% names(row_data)) return(NA_integer_)
  
  # SUBSET DATA: Filter the large dataset 'row_data' to get only rows matching the current 'admin_id'.
  # IMPROVEMENT: This is a performance bottleneck. Filtering the full 'row_data' inside the function 
  # for every single row of 'seasonality_wide_dt' (via mapply) is very slow. 
  # A data.table join or aggregation (using `by = admin_id`) would be significantly faster.
  admin_row_data <- row_data[get(admin_col) == admin_id]
  
  # Check if we actually found data for this admin unit.
  if (nrow(admin_row_data) == 0) return(NA_integer_)
  
  # FILTER FOR START MONTHS: Look at the rows for this admin. 
  # Keep only the rows where the seasonality flag is 1 (meaning a season starts there).
  # Extract the 'month' value from these rows.
  seasonal_months <- admin_row_data[get(seasonality_row_col) == 1, get(month_column)]
  
  # Check if any seasonal start months were found.
  if (length(seasonal_months) == 0) return(NA_integer_)
  
  # FIND MODE (Most Frequent Month):
  # 1. Create a frequency table of the start months found above.
  month_counts <- table(seasonal_months)
  
  # 2. Identify the month with the highest count.
  # NOTE: 'which.max' picks the first maximum in case of a tie. If month 1 and month 5 
  # appear equally often, the code will pick the one that appears first in the table.
  most_frequent_month <- as.integer(names(month_counts)[which.max(month_counts)])
  
  return(most_frequent_month)
}

In [ ]:
# EXECUTION: Apply the function to every row in 'seasonality_wide_dt'.
# This creates a new column (defined by 'season_start_month_col') with the results.
seasonality_wide_dt[, (season_start_month_col) := mapply(
  compute_start_month,
  admin_id = get(admin_id_col), # 'ADM2_ID'
  block_duration = get(season_duration_col), # `SEASONAL_BLOCK_DURATION_RAINFALL` (can be hard-coded!)
  MoreArgs = list(
    row_data = row_seasonality_dt, # output of step 1 ("is this month a start?")
    admin_col = admin_id_col, # 'ADM2_ID'
    year_column = year_col,   # 'YEAR'
    month_column = month_col  # 'MONTH'
    )
)]

head(seasonality_wide_dt)

In [ ]:
# Set SEASONAL_BLOCK_START_MONTH to NA for non-seasonal admin units
seasonality_wide_dt[get(seasonality_col) == 0 | is.na(get(seasonality_col)), (season_start_month_col) := NA_integer_]

head(seasonality_wide_dt)

------------

## Add ADM cols before exporting

In [ ]:
# add remaining admin unit columns and save the final results
admin_seasonality_wide_dt <- merge.data.table(admin_data, seasonality_wide_dt, by = c(admin_id_col), all = TRUE)
admin_seasonality_wide_dt <- admin_seasonality_wide_dt[, .SD, .SDcols = c(common_cols, seasonality_cols)]
# head(admin_seasonality_wide_dt)

In [ ]:
# Export admin_data as csv
fwrite(admin_data, file.path("~/workspace/ad_hoc_analyses/rainfall_seasonality", "admin_data.csv"))

In [ ]:
head(admin_seasonality_wide_dt)

**Save output**

`admin_seasonality_wide_dt`

In [ ]:
# Create the filename
file_stem <- paste(COUNTRY_CODE, type_of_seasonality, 'seasonality', sep = '_') # these are the filenames which will be saved in the dataset (if change, will not be available to reporting nb)
# file_stem <- paste0(COUNTRY_CODE, "_", type_of_seasonality, "_seasonality", SUFFIX) # 
filename_csv = glue("{file_stem}.csv")
filename_parquet = glue("{file_stem}.parquet")
fwrite(admin_seasonality_wide_dt, file.path(OUTPUT_DATA_PATH, filename_csv))
write_parquet(admin_seasonality_wide_dt, file.path(OUTPUT_DATA_PATH, filename_parquet))
log_msg(paste0("Rainfall seasonality results saved in folder ", OUTPUT_DATA_PATH))